# MoodFit V2 — 클래스별 epoch당 400회 균형 학습

커널을 재시작한 뒤 위에서부터 순서대로 실행하세요. 학습은 CUDA GPU를 사용합니다.

- 원본을 클래스 비율을 유지해 학습 80%·검증 20%로 먼저 분리합니다.
- 학습은 24개 무드 × 400회 = **9,600회/epoch**, 배치 32 기준 **300배치/epoch**입니다.
- 원본이 많은 무드는 **매 epoch 중복 없이 새로운 400장**을 선택합니다.
- 원본이 부족한 무드는 학습 원본을 모두 포함하고 반복 선택하며 읽을 때마다 증강합니다.
- 같은 원본 사진이 반복될 수 있으므로 400회는 독립적인 의상 400장을 의미하지 않습니다.
- 검증에는 증강을 적용하지 않습니다. 같은 착장의 서로 다른 사진은 별도 그룹 분리가 필요합니다.
- 이전 모델과 비교할 때는 동일한 검증 분할을 사용하세요.
- 저장 경로: `Model/fashion_mood_model_pytorch_v2.pth` (기존 V1 모델과 구분)

샘플러는 메인 프로세스에서 인덱스를 생성하므로 persistent worker를 사용해도 새 목록이 반영됩니다.

In [1]:
#0. 모듈 및 GPU 설정
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 사용할 수 없습니다. PyTorch CUDA 환경과 커널을 확인하세요.")

device = torch.device("cuda:0")
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
print(f"현재 사용하는 디바이스: {device}")
print(f"GPU 장치명: {torch.cuda.get_device_name(0)}")

현재 사용하는 디바이스: cuda:0
GPU 장치명: NVIDIA GeForce RTX 3060


In [ ]:
#3. V2 학습: 24개 클래스 × 800회 = 19,200회/epoch
from sklearn.metrics import balanced_accuracy_score, f1_score
from pathlib import Path

epochs = 100
for history_name in ("train_acc_history", "val_acc_history", "train_loss_history", "val_loss_history", "val_macro_f1_history", "val_balanced_acc_history"):
    globals().setdefault(history_name, [])
start_epoch = len(train_acc_history)
log_path = Path("Model/v2_training_log_target800.txt")
log_path.parent.mkdir(exist_ok=True)
if not log_path.exists():
    log_path.write_text("V2 training log (target_per_class=800)\n" + "=" * 90 + "\n", encoding="utf-8")

for epoch in range(start_epoch, epochs):
    # 시드 + epoch 값으로 각 클래스의 800장을 매번 새로 선택한다.
    train_sampler.set_epoch(epoch)
    mood_model.train()
    # 백본 가중치는 동결되어 있으므로 stochastic depth도 비활성화한다.
    for module in feature_modules[:-3]:
        module.eval()
    for module in feature_modules[-3:]:
        module.train()
    mood_model.classifier.train()
    running_loss = 0.0
    correct_train = total_train = 0
    train_bar = tqdm(
        train_loader, desc=f"Epoch {epoch + 1}/{epochs} [Train]", unit="batch",
    )
    for batch_index, (inputs, labels) in enumerate(train_bar):
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if epoch == 0 and batch_index == 0:
            print(f"model_device={next(mood_model.parameters()).device}, "
                  f"input_device={inputs.device}, batch_shape={tuple(inputs.shape)}")

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
            outputs = mood_model(inputs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * labels.size(0)
        total_train += labels.size(0)
        correct_train += (outputs.argmax(1) == labels).sum().item()
        train_bar.set_postfix(
            loss=f"{loss.item():.4f}", acc=f"{correct_train / total_train:.1%}",
        )

    if total_train != len(train_sampler):
        raise RuntimeError("클래스별 학습 횟수가 설정과 일치하지 않습니다.")
    train_loss_history.append(running_loss / total_train)
    train_acc_history.append(correct_train / total_train)

    # 검증 데이터는 원본 비율을 유지하며 무작위 증강/반복 선택하지 않는다.
    mood_model.eval()
    val_running_loss = 0.0
    correct_val = total_val = 0
    actual_labels, predicted_labels = [], []
    val_bar = tqdm(
        val_loader, desc=f"Epoch {epoch + 1}/{epochs} [Valid]", unit="batch",
    )
    with torch.inference_mode():
        for inputs, labels in val_bar:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                outputs = mood_model(inputs)
                loss = criterion(outputs, labels)
            predicted = outputs.argmax(1)
            val_running_loss += loss.item() * labels.size(0)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            actual_labels.extend(labels.cpu().tolist())
            predicted_labels.extend(predicted.cpu().tolist())
            val_bar.set_postfix(acc=f"{correct_val / total_val:.1%}")

    val_loss_history.append(val_running_loss / total_val)
    val_acc_history.append(correct_val / total_val)
    val_macro_f1_history.append(f1_score(
        actual_labels, predicted_labels, labels=list(range(num_classes)),
        average="macro", zero_division=0,
    ))
    val_balanced_acc_history.append(balanced_accuracy_score(actual_labels, predicted_labels))
    print(
        f"Epoch [{epoch + 1}/{epochs}] | "
        f"Train Loss: {train_loss_history[-1]:.4f} | "
        f"Train Acc: {train_acc_history[-1]:.2%} | "
        f"Val Loss: {val_loss_history[-1]:.4f} | "
        f"Val Acc: {val_acc_history[-1]:.2%} | "
        f"Val Macro F1: {val_macro_f1_history[-1]:.4f} | "
        f"Val Balanced Acc: {val_balanced_acc_history[-1]:.2%}", 
        flush=True,
    )
    with log_path.open("a", encoding="utf-8") as log_file:
        log_file.write(
            f"Epoch [{epoch + 1}/{epochs}] | "
            f"Train Loss: {train_loss_history[-1]:.4f} | "
            f"Train Acc: {train_acc_history[-1]:.2%} | "
            f"Val Loss: {val_loss_history[-1]:.4f} | "
            f"Val Acc: {val_acc_history[-1]:.2%} | "
            f"Val Macro F1: {val_macro_f1_history[-1]:.4f} | "
            f"Val Balanced Acc: {val_balanced_acc_history[-1]:.2%}\n"
        )

print("V2 학습 완료!")

Epoch 12/100 [Train]:   0%|          | 0/600 [00:11<?, ?batch/s]

Epoch 12/100 [Valid]:   0%|          | 0/757 [00:11<?, ?batch/s]

Epoch [12/100] | Train Loss: 1.1989 | Train Acc: 63.20% | Val Loss: 2.1967 | Val Acc: 28.09% | Val Macro F1: 0.1887 | Val Balanced Acc: 34.91%


Epoch 13/100 [Train]:   0%|          | 0/600 [00:00<?, ?batch/s]

Epoch 13/100 [Valid]:   0%|          | 0/757 [00:00<?, ?batch/s]

Epoch [13/100] | Train Loss: 1.1780 | Train Acc: 63.72% | Val Loss: 2.1864 | Val Acc: 28.42% | Val Macro F1: 0.1929 | Val Balanced Acc: 34.83%


... (18 outputs left out)

Epoch 20/100 [Train]:   0%|          | 0/600 [00:00<?, ?batch/s]

Epoch 20/100 [Valid]:   0%|          | 0/757 [00:00<?, ?batch/s]

Epoch [20/100] | Train Loss: 1.0440 | Train Acc: 67.69% | Val Loss: 2.1027 | Val Acc: 31.38% | Val Macro F1: 0.2110 | Val Balanced Acc: 35.50%


Epoch 21/100 [Train]:   0%|          | 0/600 [00:00<?, ?batch/s]

In [6]:
# ConvNeXt의 마지막 특징 추출 블록만 학습 가능하게 설정
feature_modules = list(mood_model.features.children())
unfrozen_modules = feature_modules[-3:]
for module in unfrozen_modules:
    for param in module.parameters():
        param.requires_grad = True

# 백본은 작은 학습률, 분류기는 조금 큰 학습률 사용
backbone_params = [
    param for param in mood_model.features.parameters()
    if param.requires_grad
]
optimizer = optim.AdamW([
    {
        "params": backbone_params,
        "lr": 3e-6,
    },
    {
        "params": mood_model.classifier.parameters(),
        "lr": 1e-4,
    },
], weight_decay=1e-4)

In [ ]:
#로그 저장 셀
from pathlib import Path
from datetime import datetime

required = [
    "train_loss_history",
    "train_acc_history",
    "val_loss_history",
    "val_acc_history",
    "val_macro_f1_history",
    "val_balanced_acc_history",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"먼저 학습 셀을 실행해야 합니다. 누락된 변수: {missing}"
    )

log_dir = Path("Model")
log_dir.mkdir(exist_ok=True)

log_path = log_dir / "v2_training_log.txt"

with open(log_path, "a", encoding="utf-8") as f:
    f.write("MoodFit V2 Training Log\n")
    f.write("=" * 90 + "\n")
    f.write(f"저장 시간: {datetime.now():%Y-%m-%d %H:%M:%S}\n")
    f.write(f"완료 epoch: {len(train_acc_history)}\n\n")

    f.write(
        "Epoch | Train Loss | Train Acc | Val Loss | Val Acc | "
        "Macro F1 | Balanced Acc\n"
    )
    f.write("-" * 90 + "\n")

    for i in range(len(train_acc_history)):
        f.write(
            f"{i + 1:5d} | "
            f"{train_loss_history[i]:10.4f} | "
            f"{train_acc_history[i]:9.2%} | "
            f"{val_loss_history[i]:8.4f} | "
            f"{val_acc_history[i]:7.2%} | "
            f"{val_macro_f1_history[i]:8.4f} | "
            f"{val_balanced_acc_history[i]:14.4f}\n"
        )

print(f"로그 저장 완료: {log_path}")

In [ ]:
from PIL import Image

def predict_image(img_path):
    # 1. 이미지 RGB 변환
    img = Image.open(img_path).convert('RGB')
    
    # 2. 전처리, 4차원 텐서로 변환
    img_tensor = data_transforms(img).unsqueeze(0).to(device)
    
    # 3. 모델에게 예측 요청
    mood_model.eval()
    with torch.inference_mode():
        outputs = mood_model(img_tensor)
        prob = torch.softmax(outputs, dim=1)

        confidence, idx = torch.max(prob, dim=1) #가장 높은 것 점수, 인덱스
        
        pred_mood_name = class_names[idx.item()] # 분류명
        confidence_score = confidence.item()     # 신뢰점수 (확률값)

    print(f'예측: {pred_mood_name} ({confidence_score:.2f})')

In [ ]:
#5. V2 모델 저장 (기존 V1 파일과 구분)
from pathlib import Path
import json

if not globals().get("train_acc_history"):
    raise RuntimeError("V2 학습 셀을 먼저 실행한 뒤 모델을 저장해 주세요.")

os.makedirs("Model", exist_ok=True)
model_path = Path("Model/fashion_mood_model_pytorch2_before_more_layers.pth")
torch.save(mood_model.state_dict(), model_path)

# 가중치와 클래스 순서/학습 조건을 함께 보관한다.
model_path.with_suffix(".json").write_text(
    json.dumps({
        "class_names": class_names,
        "target_per_class_per_epoch": TARGET_PER_CLASS,
        "seed": SEED,
        "epochs_completed": len(train_acc_history),
        "sampling": "resample_each_epoch",
        "unfrozen_feature_modules": 1,
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"V2 모델 저장 완료: {model_path}")

In [ ]:
from pathlib import Path
from IPython.display import Image as ShowImage, display

# 내려받은 의상 사진 10장을 차례로 예측
for path in sorted(Path("test_assets/internet_outfits/images").glob("*.jpg")):
    print(f"\n테스트 이미지: {path.name}")
    display(ShowImage(filename=str(path), width=200))
    predict_image(str(path))

In [ ]:
import numpy as np
import pandas as pd
from torchvision import datasets
from IPython.display import display, HTML

# 이미지 파일을 읽어 학습하지 않고, 폴더별 개수만 집계
dataset_info = datasets.ImageFolder(
    "CNN_train_test/CNN_train_test_image"
)

counts = np.bincount(
    dataset_info.targets,
    minlength=len(dataset_info.classes)
)

stats = pd.DataFrame({
    "무드": dataset_info.classes,
    "이미지 수": counts,
    "비율": counts / counts.sum() * 100,
})

# 표시 형식 지정
formatted = stats.copy()
formatted["이미지 수"] = formatted["이미지 수"].map("{:,}".format)
formatted["비율"] = formatted["비율"].map("{:.2f}%".format)

# 좌우 두 묶음으로 배치
middle = (len(formatted) + 1) // 2
table = pd.concat([
    formatted.iloc[:middle].reset_index(drop=True),
    formatted.iloc[middle:].reset_index(drop=True),
], axis=1).fillna("")

display(HTML(table.to_html(index=False)))

print(f"총 이미지: {counts.sum():,}장")
print(f"클래스: {len(counts)}개")
print(f"클래스별 이미지 수 중앙값: {np.median(counts):,.0f}장")

In [ ]:
import matplotlib.pyplot as plt

# 1. 그래프 크기 및 스타일 설정 (보고서 제출용 고해상도 느낌)
plt.figure(figsize=(9, 5), dpi=100)

# 2. 선 스타일 개선 (마커 추가 및 부드러운 곡선/직선 연출)
plt.plot(
    train_acc_history, 
    label='Train Accuracy', 
    color='#2b5c8f',       # 세련된 진한 파란색
    linewidth=2.5, 
    marker='o',            # 각 에포크마다 동그란 마커 표시
    markersize=5
)

plt.plot(
    val_acc_history, 
    label='Validation Accuracy', 
    color='#e74c3c',       # 눈에 잘 띄는 코랄레드 색상
    linewidth=2.5, 
    marker='s',            # 검증은 네모난 마커 표시
    markersize=5
)

# 3. 타이틀 및 라벨 디자인 (폰트 크기 및 여백 조정)
plt.title('ConvNeXt-Tiny V2 Accuracy (400 samples per class per epoch)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Epochs', fontsize=12, labelpad=10)
plt.ylabel('Accuracy', fontsize=12, labelpad=10)

# 4. Y축 범위를 0.0 ~ 1.0 (또는 0% ~ 100%)로 보기 좋게 고정하거나 여유 두기
# 만약 Accuracy가 0~1 사이 값이라면 아래 주석을 해제하세요.
# plt.ylim(0.0, 1.05)

# 5. 그리드 및 범례(Legend) 스타일링
plt.grid(True, linestyle='--', alpha=0.6)  # 점선 그리드 적용
plt.legend(fontsize=11, loc='lower right', frameon=True, shadow=True)

# 6. 레이아웃 최적화 및 출력
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 한글 표시
plt.rcParams["font.family"] = "Malgun Gothic"

# 검증 데이터에서 최대 10장 가져오기
images, labels = next(iter(val_loader))
images, labels = images[:10], labels[:10]

# 모델 예측: 출력 점수를 softmax 확률로 변환
mood_model.eval() 
with torch.inference_mode():
    outputs = mood_model(images.to(device))
    confidences, predictions = torch.softmax(outputs, dim=1).max(dim=1)

predictions = predictions.cpu()
confidences = confidences.cpu()

# 학습 전처리의 정규화를 되돌려 원래 색상으로 표시
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
images = (images.cpu() * std + mean).clamp(0, 1)

# 2행 5열 시각화
fig, axes = plt.subplots(2, 5, figsize=(16, 8))

for idx, ax in enumerate(axes.flat):
    ax.axis("off")
    if idx >= len(images):
        continue

    # RGB 이미지는 (채널, 높이, 너비) → (높이, 너비, 채널)
    ax.imshow(images[idx].permute(1, 2, 0).numpy())

    actual = class_names[labels[idx].item()]
    predicted = class_names[predictions[idx].item()]
    color = "green" if actual == predicted else "red"

    ax.set_title(
        f"원래 라벨: {actual}\n예측 라벨: {predicted} ({confidences[idx].item():.1%})",
        color=color,
        fontsize=10,
    )

plt.tight_layout()
plt.show()

In [ ]:
# 현재 V2 모델 별도 저장
from pathlib import Path
import json

if not globals().get("train_acc_history"):
    raise RuntimeError("먼저 V2 학습 셀을 실행하세요.")

completed_epoch = len(train_acc_history)
model_path = Path(f"Model/fashion_mood_model_pytorch2_epoch{completed_epoch}_last3features.pth")
model_path.parent.mkdir(exist_ok=True)
torch.save(mood_model.state_dict(), model_path)
model_path.with_suffix(".json").write_text(
    json.dumps({
        "class_names": class_names,
        "target_per_class_per_epoch": TARGET_PER_CLASS,
        "epochs_completed": completed_epoch,
        "unfrozen_feature_modules": 3,
        "best_val_macro_f1": max(val_macro_f1_history),
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"현재 V2 모델 저장 완료: {model_path}")